# ⚡ SofaScore Multi-Liga Scraper v3 — CORREGIDO
### Rayo Vallecano Scout IA — TFM Big Data Deportivo

Versión corregida con los siguientes **bugs solucionados**:

| # | Bug | Solución |
|---|-----|----------|
| 1 | `MergePestanas` usaba `outer join` acumulando filas duplicadas por variantes de nombre | Sustituido por merge con deduplicación + `groupby first` |
| 2 | `Goals`, `xG` y `Total shots` aparecían con valores cap (2.5 / 3.1 / 39.5) porque columnas repetidas entre pestañas se promediaban al hacer merge | Se eliminan duplicados de columna antes del merge usando la fuente prioritaria |
| 3 | 3 ligas (`laliga2`, `ligue2`, `liga_argentina`) con `xG = 0` en lugar de `NaN` | Si SofaScore no tiene xG para una liga, se deja `NaN` explícito |
| 4 | El merge entre pestañas usaba `outer` para `attack+defense+passing` generando filas fantasma | Ahora se usa `left join` tomando `attack` como base para no multiplicar filas |
| 5 | Sin validación post-merge: valores imposibles pasaban sin avisar | Añadido `validar_merge()` con prints de alerta para valores cap, duplicados y xG incoherente |

```
Celda 1 → Instalar dependencias
Celda 2 → Importar librerías
Celda 3 → Configurar ligas y pestañas
Celda 4 → Utilidades (normalización, logs, VALIDACIÓN)
Celda 5 → Clase Checkpoint
Celda 6 → Clase MergePestanas (CORREGIDA)
Celda 7 → Clase SofaScoreScraper
Celda 8 → ▶️ EJECUTAR
Celda 9 → Verificar resultados
```

## Celda 1 — Instalación de dependencias

In [2]:
import subprocess, sys

paquetes = ["selenium", "webdriver-manager", "pandas", "openpyxl"]

for paquete in paquetes:
    print(f"Instalando {paquete}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", paquete, "-q"])

print("\n✅ Todas las dependencias instaladas correctamente")

Instalando selenium...
Instalando webdriver-manager...
Instalando pandas...
Instalando openpyxl...

✅ Todas las dependencias instaladas correctamente


## Celda 2 — Importar librerías

In [4]:
import time
import random
import json
import os
import re
import unicodedata
import pandas as pd
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## Celda 3 — Configuración de ligas y pestañas

> **Cómo añadir una liga nueva:**
> 1. Ve a SofaScore → navega a la liga → haz clic en la pestaña **Stats**
> 2. Copia la URL completa de la barra del navegador
> 3. Añade una entrada nueva al diccionario `LIGAS`
> 4. En la Celda 8 añade la clave a `LIGAS_A_SCRAPEAR`

In [5]:
LIGAS = {
    "serie_a":       {"nombre": "Serie A",                  "url": "https://www.sofascore.com/football/tournament/italy/serie-a/23#id:76457,tab:stats"},
    "laliga":        {"nombre": "LaLiga EA Sports",          "url": "https://www.sofascore.com/football/tournament/spain/laliga/8#id:77559,tab:stats"},
    "laliga2":       {"nombre": "LaLiga Hypermotion",        "url": "https://www.sofascore.com/football/tournament/spain/laliga-2/54#id:77558,tab:stats"},
    "premier":       {"nombre": "Premier League",            "url": "https://www.sofascore.com/football/tournament/england/premier-league/17#id:76986,tab:stats"},
    "championship":  {"nombre": "Championship",              "url": "https://www.sofascore.com/football/tournament/england/championship/18#id:77347,tab:stats"},
    "bundesliga":    {"nombre": "Bundesliga",                "url": "https://www.sofascore.com/football/tournament/germany/bundesliga/35#id:77333,tab:stats"},
    "bundesliga2":   {"nombre": "2. Bundesliga",             "url": "https://www.sofascore.com/football/tournament/germany/2-bundesliga/44#id:77354,tab:stats"},
    "ligue1":        {"nombre": "Ligue 1",                   "url": "https://www.sofascore.com/football/tournament/france/ligue-1/34#id:77356,tab:stats"},
    "ligue2":        {"nombre": "Ligue 2",                   "url": "https://www.sofascore.com/football/tournament/france/ligue-2/182#id:77357,tab:stats"},
    "serie_b":       {"nombre": "Serie B",                   "url": "https://www.sofascore.com/football/tournament/italy/serie-b/53#id:79502,tab:stats"},
    "mls":           {"nombre": "MLS",                       "url": "https://www.sofascore.com/football/tournament/usa/mls/242#id:58766,tab:stats"},
    "liga_argentina":{"nombre": "Liga Profesional Argentina","url": "https://www.sofascore.com/football/tournament/argentina/liga-profesional-de-futbol/155#id:87913,tab:stats"},
}

# Pestañas de SofaScore — data-testid iguales en todas las ligas
PESTANAS = {
    "General":     "tab-summary",
    "Attack":      "tab-attack",
    "Defense":     "tab-defence",
    "Passing":     "tab-passing",
    "Goalkeeping": "tab-goalkeeper",
    "Detailed":    "tab-detailed",
}

# Fuente prioritaria por columna cuando aparece en varias pestañas.
# La pestaña listada aquí "gana"; el resto se descarta en el merge.
# Esto evita que Goals de Attack y Goals de General se promedien.
COLUMNA_FUENTE_PRIORITARIA = {
    "Goals":                   "Attack",    # Attack tiene también xG y shots
    "Expected goals (xG)":     "Attack",
    "Succ. dribbles":          "Attack",
    "Tackles":                 "Defense",   # Defense tiene interceptions, clearances
    "Assists":                 "Passing",   # Passing tiene key passes y big chances
    "Accurate passes %":       "Passing",
    "Average Sofascore Rating": "General",  # Rating global: solo el del General
}

# Pestañas que se descartan completamente del merge (subconjuntos)
PESTANAS_DESCARTAR = {"Detailed", "General"}  # General se usa solo para Rating y Team

print(f"✅ {len(LIGAS)} ligas configuradas")
print(f"✅ {len(PESTANAS)} pestañas configuradas: {list(PESTANAS.keys())}")
print(f"✅ Columnas con fuente prioritaria definida: {list(COLUMNA_FUENTE_PRIORITARIA.keys())}")

✅ 12 ligas configuradas
✅ 6 pestañas configuradas: ['General', 'Attack', 'Defense', 'Passing', 'Goalkeeping', 'Detailed']
✅ Columnas con fuente prioritaria definida: ['Goals', 'Expected goals (xG)', 'Succ. dribbles', 'Tackles', 'Assists', 'Accurate passes %', 'Average Sofascore Rating']


## Celda 4 — Utilidades (normalización, logs, validación)

In [6]:
# ══════════════════════════════════════════════════════════════════
# NORMALIZACIÓN DE NOMBRES
# ══════════════════════════════════════════════════════════════════

def normalizar_nombre(nombre: str) -> str:
    """
    'Ángel Di María' → 'angel di maria'
    Elimina tildes, convierte a minúsculas y colapsa espacios extra.
    """
    if not isinstance(nombre, str):
        return ""
    nfkd = unicodedata.normalize("NFKD", nombre)
    sin_tildes = "".join(c for c in nfkd if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", sin_tildes.lower().strip())


def log(mensaje: str, nivel: str = "INFO"):
    ts = datetime.now().strftime("%H:%M:%S")
    iconos = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERROR": "❌", "STEP": "🔷"}
    print(f"[{ts}] {iconos.get(nivel, '')} {mensaje}")


# ══════════════════════════════════════════════════════════════════
# VALIDACIÓN POST-MERGE
# ══════════════════════════════════════════════════════════════════
# BUG CORREGIDO #5: sin validación, valores imposibles pasaban silenciosamente.
# Esta función se llama después de cada merge para detectar problemas.

def validar_merge(df: pd.DataFrame, liga: str) -> None:
    """
    Comprueba el DataFrame resultante del merge y avisa de:
      - Filas duplicadas (mismo nombre + liga)
      - Valores cap en Goals / xG / Total shots (indicio de merge mal hecho)
      - xG = 0 para TODOS los jugadores de la liga (SofaScore no tiene xG)
      - xG negativo (imposible)
      - Jugadores con xG > Goals * 3 de forma masiva (incoherencia estadística)
    """
    separador = "─" * 60
    errores = 0

    print(f"\n{separador}")
    print(f"  🔍 VALIDACIÓN POST-MERGE: {liga.upper()}")
    print(separador)

    # 1. Duplicados
    n_dup = df.duplicated(subset=["Name"], keep=False).sum()
    if n_dup > 0:
        print(f"  ⚠️  BUG DETECTADO — {n_dup} filas con nombre duplicado.")
        print(f"       Causa probable: outer join entre pestañas con variantes de nombre.")
        print(f"       Acción: se aplicará deduplicación automática (ver MergePestanas).")
        ejemplos = df[df.duplicated(subset=["Name"], keep=False)]["Name"].unique()[:5]
        print(f"       Ejemplos: {list(ejemplos)}")
        errores += 1
    else:
        print(f"  ✅  Sin duplicados de nombre.")

    # 2. Goals — valor cap
    if "Goals" in df.columns:
        goals_max = df["Goals"].max()
        # Un goleador de temporada completa suele tener 15-30 goles.
        # Si el máximo es ≤ 5, es muy probable que sean datos por partido o truncados.
        if goals_max <= 5:
            n_cap = (df["Goals"] == goals_max).sum()
            print(f"  ⚠️  BUG DETECTADO — Goals máximo = {goals_max} ({n_cap} jugadores en el tope).")
            print(f"       Causa probable: columna Goals de dos pestañas distintas se fusionó mal,")
            print(f"       o SofaScore está mostrando la media por partido en lugar del total.")
            print(f"       Acción: revisar que Attack sea la fuente única de Goals.")
            errores += 1
        else:
            print(f"  ✅  Goals máximo = {goals_max:.0f} — rango plausible.")

    # 3. xG — toda la liga a 0
    if "Expected goals (xG)" in df.columns:
        todos_cero = (df["Expected goals (xG)"].fillna(0) == 0).all()
        if todos_cero:
            print(f"  ⚠️  AVISO — Expected goals (xG) = 0 para TODOS los jugadores de {liga}.")
            print(f"       Causa probable: SofaScore no publica xG para esta liga/temporada,")
            print(f"       o la pestaña Attack no se scrapeó correctamente.")
            print(f"       Acción: se sustituyen los 0 por NaN (no confundir 'sin dato' con '0 xG').")
            # Corrección automática: 0 → NaN cuando TODA la liga tiene 0
            df["Expected goals (xG)"] = df["Expected goals (xG)"].replace(0, float("nan"))
            print(f"       ✅  Corrección aplicada: xG = 0 → NaN en {liga}.")
            errores += 1

        xg_neg = (df["Expected goals (xG)"] < 0).sum()
        if xg_neg > 0:
            print(f"  ⚠️  BUG DETECTADO — {xg_neg} jugadores con xG negativo (imposible).")
            errores += 1

        xg_max = df["Expected goals (xG)"].max()
        if pd.notna(xg_max) and xg_max <= 3.5 and len(df) > 50:
            n_cap = (df["Expected goals (xG)"] == xg_max).sum()
            print(f"  ⚠️  BUG DETECTADO — xG máximo = {xg_max} ({n_cap} jugadores en el tope).")
            print(f"       Un delantero top de temporada completa suele tener xG > 15.")
            print(f"       Causa probable: mismo problema que Goals (fuente mezclada o truncada).")
            errores += 1
        elif pd.notna(xg_max):
            print(f"  ✅  xG máximo = {xg_max:.2f} — rango plausible.")

    # 4. Total shots — valor cap
    if "Total shots" in df.columns:
        shots_max = df["Total shots"].max()
        if shots_max <= 40 and len(df) > 50:
            n_cap = (df["Total shots"] == shots_max).sum()
            print(f"  ⚠️  BUG DETECTADO — Total shots máximo = {shots_max} ({n_cap} jugadores en el tope).")
            print(f"       Un delantero top suele tener 100-200 disparos en temporada completa.")
            errores += 1
        else:
            print(f"  ✅  Total shots máximo = {shots_max:.0f} — rango plausible.")

    # 5. Resumen
    print(separador)
    if errores == 0:
        print(f"  ✅  Sin problemas detectados en {liga}.")
    else:
        print(f"  ⚠️  {errores} problema(s) detectado(s) en {liga}. Revisar log anterior.")
    print(separador)


# Test rápido
print("Test normalizar_nombre:")
for n in ["Ángel Di María", "VINICIUS JR.", " Bellingham  ", "Pathé Ciss"]:
    print(f"  '{n}' → '{normalizar_nombre(n)}'")
print("\n✅ Utilidades listas")

Test normalizar_nombre:
  'Ángel Di María' → 'angel di maria'
  'VINICIUS JR.' → 'vinicius jr.'
  ' Bellingham  ' → 'bellingham'
  'Pathé Ciss' → 'pathe ciss'

✅ Utilidades listas


## Celda 5 — Sistema de Checkpoint

Guarda el progreso en un JSON. Si el scraper falla, vuelve a ejecutar la Celda 8 y retomará desde la última pestaña completada.

In [7]:
class Checkpoint:
    """
    Persiste el progreso del scraping en un JSON.

    JSON guardado en: sofascore_data/{liga}/_checkpoint_{liga}.json
    """

    def __init__(self, liga_key: str, directorio: str):
        self.path     = os.path.join(directorio, f"_checkpoint_{liga_key}.json")
        self.liga_key = liga_key
        self.datos    = self._cargar()

    def _cargar(self) -> dict:
        if os.path.exists(self.path):
            with open(self.path, "r", encoding="utf-8") as f:
                datos = json.load(f)
            completadas = len(datos["pestanas_completadas"])
            log(f"Checkpoint encontrado: {completadas} pestaña(s) ya completadas → se retomarán las demás", "WARN")
            return datos
        return {
            "liga_key":             self.liga_key,
            "pestanas_completadas": [],
            "pestanas_pendientes":  list(PESTANAS.keys()),
            "archivos_guardados":   {},
            "iniciado_en":          datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "ultima_actualizacion": None,
        }

    def guardar(self):
        self.datos["ultima_actualizacion"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump(self.datos, f, ensure_ascii=False, indent=2)

    def marcar_completada(self, pestana: str, ruta_csv: str):
        if pestana not in self.datos["pestanas_completadas"]:
            self.datos["pestanas_completadas"].append(pestana)
        if pestana in self.datos["pestanas_pendientes"]:
            self.datos["pestanas_pendientes"].remove(pestana)
        self.datos["archivos_guardados"][pestana] = ruta_csv
        self.guardar()
        log(f"Checkpoint actualizado → '{pestana}' guardada", "OK")

    def esta_completada(self, pestana: str) -> bool:
        return pestana in self.datos["pestanas_completadas"]

    def get_ruta_csv(self, pestana: str) -> str | None:
        return self.datos["archivos_guardados"].get(pestana)

    def eliminar(self):
        if os.path.exists(self.path):
            os.remove(self.path)
            log("Checkpoint eliminado (scraping completado con éxito)", "OK")


print("✅ Clase Checkpoint definida")

✅ Clase Checkpoint definida


## Celda 6 — Merge de Pestañas (CORREGIDA)

### Bugs corregidos aquí:

**Bug 1 — `outer join` multiplicaba filas:** Cuando un nombre de jugador tenía una variante mínima entre pestañas (ej. `"Vinícius"` vs `"Vinicius"`), el outer join creaba dos filas para el mismo jugador. Solución: **left join usando `Attack` como base** + deduplicación final con `groupby first`.

**Bug 2 — Columnas duplicadas promediadas:** `Goals` existe en `Attack` y en `General`. En el merge anterior, si ambas llegaban al DataFrame con el mismo nombre, pandas las promediaba silenciosamente. Solución: **antes de cada merge, se eliminan de la pestaña nueva las columnas que ya existen en la base** según `COLUMNA_FUENTE_PRIORITARIA`.

**Bug 3 — xG = 0 en ligas sin datos:** Si SofaScore no publica xG para una liga, la columna `Expected goals (xG)` se rellenaba con 0 al ser numérica. Solución: **si toda la liga tiene xG = 0, se reemplaza por `NaN`** (implementado en `validar_merge`).

In [8]:
class MergePestanas:
    """
    Une los CSV de las pestañas de SofaScore en un único DataFrame por liga.

    CORRECCIONES respecto a v2:
    ──────────────────────────
    1. Usa LEFT JOIN (base = Attack) en lugar de OUTER JOIN.
       → Evita multiplicar filas cuando un nombre varía ligeramente entre pestañas.

    2. Antes de cada join elimina de la pestaña nueva las columnas que YA están
       en la base y cuya fuente prioritaria NO es la pestaña nueva.
       → Evita que Goals de General sobreescriba o se mezcle con Goals de Attack.

    3. Deduplicación final por _key: si aun así quedan duplicados (nombres muy
       distintos entre pestañas), se queda con la fila con más datos no-NaN.

    4. Llama a validar_merge() al terminar para detectar valores imposibles.

    Uso:
        merger = MergePestanas(liga_key, {"Attack": "ruta.csv", "Defense": "ruta.csv", ...})
        df     = merger.ejecutar("salida.csv")
    """

    # Columnas de identidad presentes en todas las pestañas → se incluyen solo una vez
    COLS_IDENTIDAD = {"#", "team", "name", "player", "position", "nationality", "age",
                      "appearances", "jugador", "equipo"}

    def __init__(self, liga_key: str, archivos: dict):
        """
        liga_key : clave de liga (ej: 'bundesliga')
        archivos : {nombre_pestana: ruta_csv}
                   Solo las pestañas scrapeadas; las que fallaron no se incluyen.
        """
        self.liga_key = liga_key
        self.archivos = archivos  # {NombrePestaña: ruta.csv}

    # ── Lectura y limpieza ────────────────────────────────────────

    def _leer_csv(self, ruta: str, pestana: str) -> pd.DataFrame:
        df = pd.read_csv(ruta, encoding="utf-8")
        df.columns = [c.strip() for c in df.columns]

        col_nombre = self._detectar_col_nombre(df)
        if col_nombre is None:
            log(f"No se encontró columna de jugador en '{pestana}' ({ruta})", "ERROR")
            return pd.DataFrame()

        df["_key"] = df[col_nombre].apply(normalizar_nombre)

        # Renombrar la columna de nombre a 'Name' para uniformidad
        if col_nombre != "Name":
            df = df.rename(columns={col_nombre: "Name"})

        log(f"  '{pestana}': {len(df)} jugadores, {len(df.columns)-1} columnas", "INFO")
        return df

    def _detectar_col_nombre(self, df: pd.DataFrame) -> str | None:
        candidatas = ["Player", "Name", "Jugador", "player", "name"]
        for c in candidatas:
            if c in df.columns:
                return c
        # Fallback: primera columna string con valores de longitud > 2
        for c in df.columns:
            if df[c].dtype == object:
                muestra = df[c].dropna().head(5).tolist()
                if all(isinstance(v, str) and len(v) > 2 for v in muestra):
                    return c
        return None

    # ── Filtrado de columnas antes del merge ──────────────────────

    def _cols_a_agregar(self, df_nuevo: pd.DataFrame, pestana: str,
                        cols_ya_presentes: set) -> list:
        """
        Devuelve las columnas de df_nuevo que deben agregarse a la base:
        - No están ya presentes, O
        - Están presentes pero la fuente prioritaria de esa columna ES esta pestaña.

        BUG CORREGIDO #2: en la versión anterior se omitían todas las columnas
        ya presentes, sin importar la fuente prioritaria. Esto causaba que si
        'Goals' llegaba primero desde General (descartado), se perdía el de Attack.
        Ahora la lógica es explícita: la fuente prioritaria siempre gana.
        """
        cols = []
        for c in df_nuevo.columns:
            if c in ("_key", "Name", "Team"):
                continue
            if c.lower() in self.COLS_IDENTIDAD:
                continue
            if c not in cols_ya_presentes:
                cols.append(c)
            else:
                # La columna ya existe: solo incluir si esta pestaña es la fuente prioritaria
                fuente = COLUMNA_FUENTE_PRIORITARIA.get(c)
                if fuente and fuente.lower() == pestana.lower():
                    cols.append(c)
                    log(f"  '{c}' reemplazada por fuente prioritaria '{pestana}'", "INFO")
        return cols

    # ── Merge principal ───────────────────────────────────────────

    def ejecutar(self, ruta_salida: str) -> pd.DataFrame:
        log(f"Iniciando merge de pestañas para {self.liga_key}...", "STEP")

        # ── 1. Extraer Rating y Team del General ─────────────────
        #    (General se descarta del merge principal, pero conservamos
        #     Rating global y Team de aquí porque son más fiables.)
        rating_df = pd.DataFrame()
        if "General" in self.archivos and self.archivos["General"]:
            gen = self._leer_csv(self.archivos["General"], "General")
            if not gen.empty:
                cols_gen = ["_key", "Name"]
                if "Team" in gen.columns:
                    cols_gen.append("Team")
                if "Average Sofascore Rating" in gen.columns:
                    cols_gen.append("Average Sofascore Rating")
                rating_df = gen[cols_gen].copy()
                log(f"General → {len(gen)} jugadores, Rating y Team extraídos", "OK")

        # ── 2. Orden de merge: Attack primero (base) ─────────────
        #    BUG CORREGIDO #1: antes era OUTER JOIN para todos.
        #    Ahora Attack es la base y los demás se unen con LEFT JOIN,
        #    de forma que no se crean filas fantasma por variantes de nombre.

        orden = ["Attack", "Defense", "Passing", "Goalkeeping"]
        df_final = None

        for pestana in orden:
            if pestana in PESTANAS_DESCARTAR:
                continue
            ruta = self.archivos.get(pestana)
            if not ruta:
                log(f"'{pestana}' no disponible, omitida", "WARN")
                continue

            df_nuevo = self._leer_csv(ruta, pestana)
            if df_nuevo.empty:
                continue

            if df_final is None:
                # Primera pestaña: es la base (Attack)
                df_final = df_nuevo.copy()
                log(f"Base del merge: '{pestana}' ({len(df_final)} jugadores)", "OK")
                continue

            # Columnas ya presentes (sin _key ni Name)
            cols_presentes = set(df_final.columns) - {"_key", "Name", "Team"}

            cols_nuevas = self._cols_a_agregar(df_nuevo, pestana, cols_presentes)
            if not cols_nuevas:
                log(f"'{pestana}': sin columnas nuevas que aportar, omitida", "INFO")
                continue

            df_reducido = df_nuevo[["_key"] + cols_nuevas].copy()

            # Goalkeepping: left join normal (porteros traen datos, campo-players NaN)
            # Resto: también left join (base=Attack) — CORRECCIÓN CLAVE
            # BUG CORREGIDO #1: era 'outer' para Attack+Defense+Passing
            df_final = pd.merge(df_final, df_reducido, on="_key", how="left",
                                suffixes=("", f"_{pestana.lower()}"))

            # Si quedaron columnas duplicadas con sufijo (de suffixes), eliminarlas
            cols_sufijo = [c for c in df_final.columns if c.endswith(f"_{pestana.lower()}")]
            if cols_sufijo:
                log(f"  Columnas duplicadas eliminadas de '{pestana}': {cols_sufijo}", "WARN")
                df_final = df_final.drop(columns=cols_sufijo)

            log(f"'{pestana}' mergeada → +{len(cols_nuevas)} cols | "
                f"{df_final['_key'].nunique()} jugadores únicos", "OK")

        if df_final is None or df_final.empty:
            log("No se pudo generar el DataFrame final", "ERROR")
            return pd.DataFrame()

        # ── 3. Añadir Rating y Team del General ──────────────────
        if not rating_df.empty:
            cols_rating = [c for c in rating_df.columns if c not in df_final.columns or c == "_key"]
            df_final = pd.merge(df_final,
                                rating_df[["_key"] + [c for c in cols_rating if c != "_key"]],
                                on="_key", how="left",
                                suffixes=("", "_gen"))
            # Eliminar columnas _gen duplicadas
            for c in [col for col in df_final.columns if col.endswith("_gen")]:
                df_final = df_final.drop(columns=[c])

        # ── 4. Columna es_portero ─────────────────────────────────
        if "Goalkeeping" in self.archivos and self.archivos["Goalkeeping"]:
            gk = self._leer_csv(self.archivos["Goalkeeping"], "Goalkeeping")
            if not gk.empty:
                porteros = set(gk["_key"])
                df_final["es_portero"] = df_final["_key"].isin(porteros)
                log(f"es_portero → {df_final['es_portero'].sum()} porteros marcados", "OK")

        # ── 5. Deduplicación final ────────────────────────────────
        #    BUG CORREGIDO #1 (refuerzo): si aun así quedaron duplicados,
        #    quedarse con la fila que tenga más columnas no-NaN.
        n_antes = len(df_final)
        if df_final["_key"].duplicated().any():
            log(f"⚠️  Duplicados detectados antes de deduplicar: {df_final['_key'].duplicated().sum()} filas extra.", "WARN")
            # Puntuar filas por cantidad de valores no-NaN
            df_final["_score_nonan"] = df_final.notna().sum(axis=1)
            df_final = (df_final
                        .sort_values("_score_nonan", ascending=False)
                        .drop_duplicates(subset=["_key"], keep="first")
                        .drop(columns=["_score_nonan"]))
            log(f"Deduplicación: {n_antes} → {len(df_final)} filas", "OK")

        # ── 6. Columna liga y reordenado ──────────────────────────
        df_final = df_final.drop(columns=["_key"], errors="ignore")
        df_final.insert(0, "liga", self.liga_key)

        cols_primeras = ["liga", "Name", "Team", "es_portero"]
        cols_resto    = [c for c in df_final.columns if c not in cols_primeras]
        rating_col    = ["Average Sofascore Rating"] if "Average Sofascore Rating" in cols_resto else []
        cols_stats    = [c for c in cols_resto if c not in rating_col]
        df_final      = df_final[[c for c in cols_primeras if c in df_final.columns]
                                  + cols_stats + rating_col]

        # ── 7. Guardar ────────────────────────────────────────────
        df_final.to_csv(ruta_salida, index=False, encoding="utf-8")
        log(f"CSV unificado guardado: {ruta_salida}", "OK")
        log(f"  → {len(df_final)} jugadores × {len(df_final.columns)} columnas", "OK")

        # ── 8. Validación ─────────────────────────────────────────
        validar_merge(df_final, self.liga_key)

        return df_final


print("✅ Clase MergePestanas (v3, corregida) definida")

✅ Clase MergePestanas (v3, corregida) definida


## Celda 7 — Scraper Principal

In [9]:
class SofaScoreScraper:
    """
    Scraper de SofaScore para estadísticas de jugadores.

    Parámetros:
        liga_key        : clave del diccionario LIGAS (ej: 'serie_a', 'mls')
        headless        : True = sin ventana; False = ver el navegador
        directorio_base : carpeta raíz donde se guardan los datos
    """

    def __init__(self, liga_key: str, headless: bool = True,
                 directorio_base: str = "sofascore_data"):
        if liga_key not in LIGAS:
            raise ValueError(f"Liga '{liga_key}' no configurada. Opciones: {list(LIGAS.keys())}")

        self.liga_key = liga_key
        self.liga_cfg = LIGAS[liga_key]
        self.headless = headless

        self.directorio = os.path.join(directorio_base, liga_key)
        self.dir_raw    = os.path.join(self.directorio, "raw")
        self.dir_final  = os.path.join(directorio_base, "final")
        for d in [self.directorio, self.dir_raw, self.dir_final]:
            os.makedirs(d, exist_ok=True)

        self.checkpoint = Checkpoint(liga_key, self.directorio)
        self.driver = None
        self.wait   = None

    # ── Driver ───────────────────────────────────────────────────

    def _inicializar_driver(self):
        options = webdriver.ChromeOptions()
        if self.headless:
            options.add_argument("--headless=new")
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option("useAutomationExtension", False)
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--window-size=1920,1080")
        options.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=options
        )
        self.driver.execute_script(
            "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        )
        self.wait = WebDriverWait(self.driver, 20)
        log("Driver iniciado", "OK")

    def _cerrar_driver(self):
        if self.driver:
            self.driver.quit()
            log("Driver cerrado", "OK")

    # ── Helpers ──────────────────────────────────────────────────

    def _pausa(self, min_s: float = 1.0, max_s: float = 3.0):
        time.sleep(random.uniform(min_s, max_s))

    def _scroll(self):
        self.driver.execute_script(f"window.scrollBy(0, {random.randint(100, 400)});")
        self._pausa(0.3, 0.8)

    def _limpiar_overlays(self):
        selectores = [
            ".fc-dialog-overlay", ".fc-dialog-container",
            ".modal-backdrop", "[role='dialog']", ".cookie-banner",
        ]
        eliminados = 0
        for sel in selectores:
            for el in self.driver.find_elements(By.CSS_SELECTOR, sel):
                try:
                    self.driver.execute_script("arguments[0].remove();", el)
                    eliminados += 1
                except Exception:
                    pass
        if eliminados:
            log(f"{eliminados} overlays eliminados", "OK")

    def _cerrar_cookies(self):
        self._pausa(2, 3)
        for sel in [
            "button.fc-button.fc-cta-consent.fc-primary-button",
            ".fc-cta-consent",
            "#onetrust-accept-btn-handler",
            "button.didomi-button-standard",
        ]:
            try:
                btn = self.driver.find_element(By.CSS_SELECTOR, sel)
                if btn.is_displayed():
                    btn.click()
                    self._pausa(1, 2)
                    log("Cookies aceptadas", "OK")
                    return
            except Exception:
                pass
        try:
            btn = self.driver.find_element(
                By.XPATH,
                "//button[contains(text(),'Consent') or contains(text(),'Accept') or contains(text(),'Aceptar')]",
            )
            if btn.is_displayed():
                btn.click()
                self._pausa(1, 2)
                log("Cookies aceptadas (XPath)", "OK")
        except Exception:
            pass

    def _navegar_inicio(self):
        url = self.liga_cfg["url"]
        log(f"Navegando a {url}", "STEP")
        self.driver.get(url)
        self._pausa(3, 5)
        self._cerrar_cookies()
        self._limpiar_overlays()
        self._scroll()

    def _cambiar_pestana(self, data_testid: str, nombre: str) -> bool:
        try:
            btn = self.wait.until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, f'button[data-testid="{data_testid}"]')
                )
            )
            self.driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            self._pausa(0.5, 1.0)
            try:
                self.wait.until(EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, f'button[data-testid="{data_testid}"]')
                ))
                btn.click()
            except Exception:
                self.driver.execute_script("arguments[0].click();", btn)
            self._pausa(2, 3)
            log(f"Pestaña '{nombre}' activada", "OK")
            return True
        except Exception as e:
            log(f"Error al cambiar a '{nombre}': {e}", "ERROR")
            # Segundo intento limpiando overlays
            try:
                self._limpiar_overlays()
                btn = self.driver.find_element(
                    By.CSS_SELECTOR, f'button[data-testid="{data_testid}"]'
                )
                self.driver.execute_script("arguments[0].click();", btn)
                self._pausa(2, 3)
                log(f"Pestaña '{nombre}' activada (2º intento)", "OK")
                return True
            except Exception:
                return False

    # ── Extracción de tabla ───────────────────────────────────────

    def _extraer_tabla(self) -> dict | None:
        """Extrae encabezados y filas de la tabla visible en pantalla."""
        try:
            xpath = " | ".join([
                f'//*[@id="tabpanel-{p}"]/div/table'
                for p in ["summary", "attack", "defence", "passing", "goalkeeper", "detailed"]
            ])
            tabla = self.wait.until(EC.presence_of_element_located((By.XPATH, xpath)))

            headers = []
            for th in tabla.find_elements(By.TAG_NAME, "th"):
                titulo = th.get_attribute("title")
                headers.append(titulo if titulo else th.text.strip())

            filas = []
            for tr in tabla.find_element(By.TAG_NAME, "tbody").find_elements(By.TAG_NAME, "tr"):
                fila = []
                for td in tr.find_elements(By.TAG_NAME, "td"):
                    texto = td.text.strip() or td.get_attribute("textContent").strip()
                    fila.append(texto)
                if fila:
                    filas.append(fila)

            return {"headers": headers, "filas": filas}
        except Exception as e:
            log(f"Error extrayendo tabla: {e}", "ERROR")
            return None

    # ── Paginación ────────────────────────────────────────────────

    def _siguiente_pagina(self) -> bool:
        """
        Hace clic en el botón › del paginador.
        Devuelve True si se avanzó, False si era la última página.
        """
        paneles = ["tabpanel-summary", "tabpanel-attack", "tabpanel-defence",
                   "tabpanel-passing", "tabpanel-goalkeeper", "tabpanel-detailed"]
        try:
            btn_sig = None
            for panel_id in paneles:
                try:
                    paginador = self.driver.find_element(
                        By.XPATH, f'//*[@id="{panel_id}"]/div/div'
                    )
                    botones = paginador.find_elements(
                        By.CSS_SELECTOR, "button.button--size_primary"
                    )
                    if botones:
                        btn_sig = botones[-1]  # › es el último
                        break
                except NoSuchElementException:
                    continue

            if btn_sig is None:
                return False  # pestaña de una sola página (sin paginador)

            if btn_sig.get_attribute("disabled") is not None:
                return False  # última página

            self.driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});", btn_sig
            )
            self._pausa(0.3, 0.6)
            try:
                btn_sig.click()
            except Exception:
                self.driver.execute_script("arguments[0].click();", btn_sig)

            self._pausa(2, 3)
            return True

        except Exception as e:
            log(f"Error en paginación: {e}", "WARN")
            return False

    # ── Scraping de una pestaña ───────────────────────────────────

    def _guardar_csv_pestana(self, nombre: str, headers: list, filas: list) -> str:
        df = pd.DataFrame(filas, columns=headers)
        ruta = os.path.join(self.dir_raw, f"{self.liga_key}_{nombre.lower()}.csv")
        df.to_csv(ruta, index=False, encoding="utf-8")
        log(f"CSV guardado: {ruta} ({len(df)} filas × {len(df.columns)} cols)", "OK")
        return ruta

    def _guardado_incremental(self, nombre: str, headers: list, filas: list):
        df = pd.DataFrame(filas, columns=headers)
        ruta = os.path.join(self.dir_raw,
                            f"_parcial_{self.liga_key}_{nombre.lower()}.csv")
        df.to_csv(ruta, index=False, encoding="utf-8")

    def _scrapear_pestana(self, nombre: str, data_testid: str) -> str | None:
        # Checkpoint: ¿ya completada?
        if self.checkpoint.esta_completada(nombre):
            ruta = self.checkpoint.get_ruta_csv(nombre)
            log(f"'{nombre}' ya completada (checkpoint). Saltando.", "WARN")
            return ruta

        log(f"{'─'*50}", "INFO")
        log(f"Pestaña: {nombre}", "STEP")

        if not self._cambiar_pestana(data_testid, nombre):
            return None

        todas_filas = []
        headers     = None
        num_pagina  = 1

        while True:
            log(f"  Página {num_pagina}...", "INFO")
            datos = self._extraer_tabla()
            if datos:
                if headers is None:
                    headers = datos["headers"]
                todas_filas.extend(datos["filas"])
                log(f"  Acumuladas {len(todas_filas)} filas", "INFO")

            # Guardado incremental cada 5 páginas
            if num_pagina % 5 == 0 and todas_filas and headers:
                self._guardado_incremental(nombre, headers, todas_filas)
                log(f"  Guardado incremental: {len(todas_filas)} filas hasta pág. {num_pagina}", "OK")

            # Intentar avanzar a la siguiente página
            if not self._siguiente_pagina():
                log(f"  Última página alcanzada ({num_pagina}). Fin de '{nombre}'.", "OK")
                break

            num_pagina += 1

        if not todas_filas or not headers:
            log(f"Sin datos para '{nombre}'", "WARN")
            return None

        ruta = self._guardar_csv_pestana(nombre, headers, todas_filas)
        self.checkpoint.marcar_completada(nombre, ruta)
        return ruta

    # ── Ejecución principal ───────────────────────────────────────

    def ejecutar(self) -> str | None:
        """
        Ejecuta el scraping completo:
          1. Scrapea las pestañas configuradas (con checkpoint)
          2. Merge en un único CSV unificado (con validación)
          3. Devuelve la ruta del CSV final
        """
        nombre_liga = self.liga_cfg["nombre"]
        log(f"{'═'*60}", "INFO")
        log(f"INICIANDO: {nombre_liga}", "STEP")
        log(f"{'═'*60}", "INFO")

        self._inicializar_driver()

        try:
            self._navegar_inicio()
            archivos_pestanas = {}

            for nombre, testid in PESTANAS.items():
                ruta = self._scrapear_pestana(nombre, testid)
                if ruta:
                    archivos_pestanas[nombre] = ruta
                else:
                    log(f"'{nombre}' sin datos, se omite en el merge", "WARN")
                self._pausa(2, 4)

        except Exception as e:
            log(f"Error durante el scraping: {e}", "ERROR")
            import traceback
            traceback.print_exc()
            log("Progreso guardado en checkpoint. Vuelve a ejecutar para retomar.", "WARN")
            return None
        finally:
            self._cerrar_driver()

        if not archivos_pestanas:
            log("No hay archivos para mergear", "ERROR")
            return None

        log(f"{'═'*60}", "INFO")
        log(f"MERGE FINAL: {nombre_liga}", "STEP")

        nombre_final = f"{self.liga_key}_players_unified.csv"
        ruta_final   = os.path.join(self.dir_final, nombre_final)

        # BUG CORREGIDO: ahora MergePestanas recibe liga_key para validar_merge
        merger   = MergePestanas(self.liga_key, archivos_pestanas)
        df_final = merger.ejecutar(ruta_final)

        if not df_final.empty:
            log(f"COMPLETADO: {nombre_liga}", "OK")
            log(f"CSV → {ruta_final}  ({len(df_final)} × {len(df_final.columns)} cols)", "OK")
            self.checkpoint.eliminar()
            return ruta_final
        else:
            log("El merge no generó datos", "ERROR")
            return None


print("✅ Clase SofaScoreScraper definida")

✅ Clase SofaScoreScraper definida


## Celda 8 — ▶️ EJECUTAR

> **Solo tienes que cambiar `LIGAS_A_SCRAPEAR`** con la clave de la liga que quieras.

| Clave | Liga |
|---|---|
| `serie_a` | Serie A |
| `laliga` | LaLiga EA Sports |
| `laliga2` | LaLiga Hypermotion |
| `premier` | Premier League |
| `championship` | Championship |
| `bundesliga` | Bundesliga |
| `bundesliga2` | 2. Bundesliga |
| `ligue1` | Ligue 1 |
| `ligue2` | Ligue 2 |
| `serie_b` | Serie B |
| `mls` | MLS |
| `liga_argentina` | Liga Profesional Argentina |

> Si el scraper se cae a mitad: vuelve a ejecutar esta celda sin cambiar nada — el checkpoint retomará automáticamente.

In [10]:
LIGAS_A_SCRAPEAR = [
    "serie_a",
    #"laliga",
    #"laliga2",
    #"premier",
    #"championship",
    #"bundesliga",
    "bundesliga2",
    #"ligue1",
    #"ligue2",
    "serie_b",
    "mls",
    "liga_argentina",
]

HEADLESS = True  # False → ves el navegador en tiempo real (útil para depurar)

resultados = {}

for liga_key in LIGAS_A_SCRAPEAR:
    print(f"\n{'█'*60}")
    print(f"  Iniciando: {LIGAS[liga_key]['nombre']}  ({liga_key})")
    print(f"{'█'*60}")
    try:
        scraper = SofaScoreScraper(
            liga_key        = liga_key,
            headless        = HEADLESS,
            directorio_base = "sofascore_data",
        )
        ruta = scraper.ejecutar()
        resultados[liga_key] = ruta
    except Exception as e:
        print(f"❌ Error en {liga_key}: {e}")
        resultados[liga_key] = None

print(f"\n{'═'*60}")
print("  RESUMEN FINAL")
print(f"{'═'*60}")
for liga_key, ruta in resultados.items():
    nombre = LIGAS[liga_key]['nombre']
    if ruta:
        print(f"  ✅  {nombre:<30s} → {ruta}")
    else:
        print(f"  ❌  {nombre:<30s} → FALLÓ (vuelve a ejecutar esta celda)")
print(f"{'═'*60}")


████████████████████████████████████████████████████████████
  Iniciando: Serie A  (serie_a)
████████████████████████████████████████████████████████████
[19:32:07] ⚠️  Checkpoint encontrado: 3 pestaña(s) ya completadas → se retomarán las demás
[19:32:07] ℹ️  ════════════════════════════════════════════════════════════
[19:32:07] 🔷 INICIANDO: Serie A
[19:32:07] ℹ️  ════════════════════════════════════════════════════════════
[19:32:21] ✅ Driver iniciado
[19:32:21] 🔷 Navegando a https://www.sofascore.com/football/tournament/italy/serie-a/23#id:76457,tab:stats
[19:33:03] ✅ Cookies aceptadas
[19:33:04] ⚠️  'General' ya completada (checkpoint). Saltando.
[19:33:08] ⚠️  'Attack' ya completada (checkpoint). Saltando.
[19:33:11] ⚠️  'Defense' ya completada (checkpoint). Saltando.
[19:33:14] ℹ️  ──────────────────────────────────────────────────
[19:33:14] 🔷 Pestaña: Passing
[19:33:19] ✅ Pestaña 'Passing' activada
[19:33:19] ℹ️    Página 1...
[19:33:25] ℹ️    Acumuladas 20 filas
[19:33:30] ℹ️

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MONITOR EN TIEMPO REAL — ejecuta esta celda mientras corre el scraper
# ══════════════════════════════════════════════════════════════════
#
# Lee los CSV parciales que el scraper guarda cada 5 páginas y muestra
# los últimos jugadores extraídos con todas sus columnas.
#
# USO:
#   1. Lanza el scraper en la Celda 8
#   2. Ejecuta esta celda en otra pestaña del notebook (o justo después)
#   3. Verás los datos actualizarse cada N segundos
#
# Para parar: Kernel → Interrupt (o el botón ■)
# ══════════════════════════════════════════════════════════════════

import os
import glob
import time
import pandas as pd
from IPython.display import display, clear_output

# ── Configuración ────────────────────────────────────────────────
CARPETA_BASE   = "sofascore_data"   # misma que en el scraper
INTERVALO_SEG  = 8                  # segundos entre refresco
N_JUGADORES    = 20                 # cuántos jugadores mostrar por pestaña
MOSTRAR_COLS   = None               # None = todas; o lista: ["Name","Goals","xG"]
# ─────────────────────────────────────────────────────────────────

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 25)

def leer_parcial_mas_reciente():
    """
    Devuelve un dict {nombre_archivo: DataFrame} con todos los CSV parciales
    encontrados en sofascore_data/*/raw/_parcial_*.csv
    Solo incluye los que se hayan modificado en los últimos 10 minutos.
    """
    patron = os.path.join(CARPETA_BASE, "*", "raw", "_parcial_*.csv")
    archivos = glob.glob(patron)
    resultado = {}
    ahora = time.time()
    for ruta in sorted(archivos):
        try:
            # Solo archivos modificados recientemente (scraper activo)
            if ahora - os.path.getmtime(ruta) > 600:
                continue
            df = pd.read_csv(ruta)
            if df.empty:
                continue
            nombre = os.path.basename(ruta).replace("_parcial_", "").replace(".csv", "")
            resultado[nombre] = df
        except Exception:
            pass
    return resultado

def leer_csv_finales():
    """
    Lee los CSV ya completados (sin _parcial_) para las pestañas terminadas.
    """
    patron = os.path.join(CARPETA_BASE, "*", "raw", "*.csv")
    archivos = glob.glob(patron)
    resultado = {}
    for ruta in sorted(archivos):
        nombre = os.path.basename(ruta)
        if nombre.startswith("_parcial_") or nombre.startswith("_checkpoint"):
            continue
        try:
            df = pd.read_csv(ruta)
            if df.empty:
                continue
            clave = nombre.replace(".csv", "")
            resultado[clave] = df
        except Exception:
            pass
    return resultado

def mostrar_dataframe(df: pd.DataFrame, titulo: str):
    cols = MOSTRAR_COLS if MOSTRAR_COLS else df.columns.tolist()
    cols = [c for c in cols if c in df.columns]
    df_mostrar = df[cols].tail(N_JUGADORES).reset_index(drop=True)
    print(f"\n{'━'*70}")
    print(f"  📋  {titulo}  —  {len(df)} jugadores totales (mostrando últimos {min(N_JUGADORES, len(df))})")
    print(f"{'━'*70}")
    print(df_mostrar.to_string(index=True))

def resumen_estado():
    """Muestra un estado rápido de qué pestañas están en curso o completadas."""
    checkpoints = glob.glob(os.path.join(CARPETA_BASE, "*", "_checkpoint_*.json"))
    if not checkpoints:
        print("  ℹ️  No hay scraper en curso (sin checkpoints activos)")
        return

    import json as _json
    for cp_ruta in sorted(checkpoints):
        try:
            with open(cp_ruta) as f:
                cp = _json.load(f)
            liga       = cp.get("liga_key", "?")
            completadas = cp.get("pestanas_completadas", [])
            pendientes  = cp.get("pestanas_pendientes", [])
            ultima      = cp.get("ultima_actualizacion", "—")
            print(f"  🏆 {liga:<15} | ✅ {completadas} | ⏳ {pendientes} | último: {ultima}")
        except Exception:
            pass

iteracion = 0
print("🔄 Monitor iniciado. Buscando datos en sofascore_data/*/raw/ ...")
print("   Para parar: Kernel → Interrupt\n")

while True:
    iteracion += 1
    clear_output(wait=True)

    ahora_str = time.strftime("%H:%M:%S")
    print(f"{'═'*70}")
    print(f"  🔍 MONITOR DE SCRAPING  —  {ahora_str}  (refresco #{iteracion})")
    print(f"{'═'*70}")

    # Estado de checkpoints
    print("\n📌 Estado del scraper:")
    resumen_estado()

    # CSV parciales (pestaña en curso)
    parciales = leer_parcial_mas_reciente()
    if parciales:
        print(f"\n⚡ PESTAÑA EN CURSO (datos parciales):")
        for nombre, df in parciales.items():
            mostrar_dataframe(df, f"PARCIAL — {nombre}")
    
    # CSV finales (pestañas completadas)
    finales = leer_csv_finales()
    if finales:
        print(f"\n✅ PESTAÑAS COMPLETADAS:")
        for nombre, df in finales.items():
            mostrar_dataframe(df, nombre)

    if not parciales and not finales:
        print("\n  ⏳ Esperando que el scraper genere datos...")
        print(f"     Buscando en: {os.path.abspath(CARPETA_BASE)}/*/raw/")

    print(f"\n\n  ⏱️  Próximo refresco en {INTERVALO_SEG}s  —  Ctrl+C para parar")
    time.sleep(INTERVALO_SEG)


══════════════════════════════════════════════════════════════════════
  🔍 MONITOR DE SCRAPING  —  04:17:24  (refresco #619)
══════════════════════════════════════════════════════════════════════

📌 Estado del scraper:
  🏆 liga_argentina  | ✅ ['General', 'Attack'] | ⏳ ['Defense', 'Passing', 'Goalkeeping', 'Detailed'] | último: 2026-06-09 21:38:01
  🏆 serie_a         | ✅ ['General', 'Attack', 'Defense'] | ⏳ ['Passing', 'Goalkeeping', 'Detailed'] | último: 2026-06-06 10:39:40
  🏆 serie_b         | ✅ ['General', 'Attack'] | ⏳ ['Defense', 'Passing', 'Goalkeeping', 'Detailed'] | último: 2026-06-09 21:03:31

✅ PESTAÑAS COMPLETADAS:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📋  bundesliga_attack  —  499 jugadores totales (mostrando últimos 20)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
      #  Team                 Name  Goals  Expected goals (xG)  Big chances missed  Succ. dribbles  Total shots  Goal conversion %  Average Sofascore R

## Celda 9 — Merge con Transfermarket

In [11]:
"""
merge_con_transfermarket.py (v3)
────────────────────────────────
Une los CSVs unificados de SofaScore con Transfermarket.
Sin cambios en la lógica de matching, pero ahora también llama
a validar_merge() sobre el resultado final.
"""

import glob
from difflib import SequenceMatcher

LIGA_MAP = {
    "laliga":         ["LaLiga EA Sports"],
    "laliga2":        ["LaLiga Hypermotion"],
    "premier":        ["Premier League"],
    "championship":   ["Championship"],
    "bundesliga":     ["Bundesliga"],
    "bundesliga2":    ["2. Bundesliga"],
    "serie_a":        ["Serie A"],
    "serie_b":        ["Serie B"],
    "ligue1":         ["Ligue 1"],
    "ligue2":         ["Ligue 2"],
    "mls":            ["MLS"],
    "liga_argentina": ["Liga Profesional"],
}

TM_COLS_KEEP = {
    "Club":           "tm_club",
    "Liga":           "tm_liga",
    "Posicion":       "posicion",
    "Nacimiento":     "fecha_nacimiento",
    "Edad":           "edad",
    "Nacionalidades": "nacionalidades",
    "Fin_Contrato":   "fin_contrato",
    "Valor_mercado":  "valor_mercado",
}

FUZZY_THRESHOLD = 0.88


def normalizar(nombre: str) -> str:
    if not isinstance(nombre, str):
        return ""
    nfkd = unicodedata.normalize("NFKD", nombre)
    sin_tildes = "".join(c for c in nfkd if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", sin_tildes.lower().strip())


def fuzzy_match(nombre: str, candidatos: list, threshold: float) -> str | None:
    mejor_ratio, mejor = 0.0, None
    for cand in candidatos:
        r = SequenceMatcher(None, nombre, cand).ratio()
        if r > mejor_ratio:
            mejor_ratio, mejor = r, cand
    return mejor if mejor_ratio >= threshold else None


def merge_liga_con_tm(liga: str, df_ss: pd.DataFrame,
                      df_tm: pd.DataFrame, fuzzy: bool = True) -> pd.DataFrame:
    ligas_tm = LIGA_MAP.get(liga, [])
    tm_liga  = df_tm[df_tm["Liga"].isin(ligas_tm)].copy() if ligas_tm else df_tm.copy()
    tm_liga  = tm_liga.dropna(subset=["Nombre"])

    df_ss   = df_ss.copy()
    tm_liga = tm_liga.copy()

    df_ss["_key"]   = df_ss["Name"].apply(normalizar)
    tm_liga["_key"] = tm_liga["Nombre"].apply(normalizar)
    tm_liga         = tm_liga.drop_duplicates(subset="_key", keep="first")

    cols_tm  = ["_key"] + [c for c in TM_COLS_KEEP if c in tm_liga.columns]
    tm_select = tm_liga[cols_tm].rename(columns=TM_COLS_KEEP)

    df_merged  = pd.merge(df_ss, tm_select, on="_key", how="left")
    n_total    = len(df_ss)
    n_matched  = df_merged["valor_mercado"].notna().sum()
    print(f"  Match exacto   : {n_matched:4d} / {n_total} ({n_matched/n_total*100:.1f}%)")

    if fuzzy and (n_total - n_matched) > 0:
        tm_keys    = tm_liga["_key"].tolist()
        fuzzy_map  = {}
        sin_match  = df_merged["valor_mercado"].isna()
        for ss_key in df_merged.loc[sin_match, "_key"].unique():
            tm_key = fuzzy_match(ss_key, tm_keys, FUZZY_THRESHOLD)
            if tm_key:
                fuzzy_map[ss_key] = tm_key

        if fuzzy_map:
            tm_lookup = tm_select.set_index("_key")
            for ss_key, tm_key in fuzzy_map.items():
                if tm_key not in tm_lookup.index:
                    continue
                tm_row = tm_lookup.loc[tm_key]
                mask = (df_merged["_key"] == ss_key) & sin_match
                for col in tm_row.index:
                    if col in df_merged.columns:
                        df_merged.loc[mask, col] = tm_row[col]
            print(f"  Fuzzy matched  : {len(fuzzy_map):4d} adicionales (threshold={FUZZY_THRESHOLD})")

    n_sin = n_total - df_merged["valor_mercado"].notna().sum()
    print(f"  Sin match final: {n_sin:4d} jugadores (NaN en columnas TM)")

    df_merged = df_merged.drop(columns=["_key"])

    # ── Validación post-merge con TM ──────────────────────────────
    # BUG CORREGIDO #3: si xG era 0 para toda la liga antes del merge,
    # la validación lo corrige a NaN. Se vuelve a llamar aquí por si el
    # merge con TM generó algún artefacto extra.
    validar_merge(df_merged, liga)

    # Reordenar
    cols_id    = ["liga", "Name", "tm_club", "tm_liga", "posicion", "edad",
                  "fecha_nacimiento", "nacionalidades", "fin_contrato",
                  "valor_mercado", "es_portero"]
    cols_stats = [c for c in df_merged.columns
                  if c not in cols_id and c != "Average Sofascore Rating"]
    cols_rating = ["Average Sofascore Rating"] if "Average Sofascore Rating" in df_merged.columns else []
    col_order = ([c for c in cols_id if c in df_merged.columns]
                 + [c for c in cols_stats if c in df_merged.columns]
                 + cols_rating)
    return df_merged[col_order]


def procesar_ligas_tm(ligas: list, carpeta_ss: str, ruta_tm: str,
                      carpeta_salida: str, fuzzy: bool = True):
    print(f"📂 Cargando Transfermarket ({ruta_tm})...")
    df_tm = pd.read_csv(ruta_tm)
    if "Error" in df_tm.columns:
        df_tm = df_tm[df_tm["Error"].isna()].copy()
    print(f"   {len(df_tm)} jugadores válidos en TM")

    os.makedirs(carpeta_salida, exist_ok=True)
    resultados = {}

    for liga in ligas:
        patron     = os.path.join(carpeta_ss, f"{liga}_players_unified.csv")
        candidatos = glob.glob(patron)
        if not candidatos:
            print(f"\n⚠️  No se encontró CSV para '{liga}' en {carpeta_ss}")
            resultados[liga] = "❌ Sin CSV"
            continue

        df_ss = pd.read_csv(candidatos[0])
        print(f"\n{'─'*55}")
        print(f"  Liga: {liga.upper()}  ({len(df_ss)} jugadores SS)")
        print(f"{'─'*55}")

        try:
            df_final  = merge_liga_con_tm(liga, df_ss, df_tm, fuzzy=fuzzy)
            ruta_out  = os.path.join(carpeta_salida, f"{liga}_master.csv")
            df_final.to_csv(ruta_out, index=False, encoding="utf-8")
            print(f"  💾 {ruta_out}  ({len(df_final)} × {len(df_final.columns)} cols)")
            resultados[liga] = f"✅ {ruta_out}"
        except Exception as e:
            import traceback
            print(f"  ❌ Error: {e}")
            traceback.print_exc()
            resultados[liga] = f"❌ Error: {e}"

    print(f"\n{'═'*55}")
    print("  RESUMEN FINAL MERGE TM")
    print(f"{'═'*55}")
    for liga, estado in resultados.items():
        print(f"  {liga:<20s} → {estado}")
    print(f"{'═'*55}")


# ── CONFIGURACIÓN — ajusta estas rutas ──────────────────────────
SOFASCORE_FINAL  = "sofascore_data/final"
RUTA_TM          = "data_notebook/jugadores_transfermarket.csv"
CARPETA_SALIDA   = "dataset_final"
FUZZY            = True

csvs = glob.glob(os.path.join(SOFASCORE_FINAL, "*_players_unified.csv"))
ligas_detectadas = [os.path.basename(c).replace("_players_unified.csv", "") for c in csvs]

if ligas_detectadas:
    print(f"🔍 Ligas detectadas: {ligas_detectadas}")
    procesar_ligas_tm(ligas_detectadas, SOFASCORE_FINAL, RUTA_TM, CARPETA_SALIDA, FUZZY)
else:
    print(f"❌ No hay CSVs en '{SOFASCORE_FINAL}'. Ejecuta la Celda 8 primero.")

🔍 Ligas detectadas: ['bundesliga2', 'bundesliga', 'championship', 'laliga2', 'laliga', 'liga_argentina', 'ligue1', 'ligue2', 'mls', 'premier', 'serie_a', 'serie_b']
📂 Cargando Transfermarket (data_notebook/jugadores_transfermarket.csv)...
   7205 jugadores válidos en TM

───────────────────────────────────────────────────────
  Liga: BUNDESLIGA2  (536 jugadores SS)
───────────────────────────────────────────────────────
  Match exacto   :  266 / 536 (49.6%)
  Fuzzy matched  :   15 adicionales (threshold=0.88)
  Sin match final:  256 jugadores (NaN en columnas TM)

────────────────────────────────────────────────────────────
  🔍 VALIDACIÓN POST-MERGE: BUNDESLIGA2
────────────────────────────────────────────────────────────
  ✅  Sin duplicados de nombre.
  ✅  Goals máximo = 19 — rango plausible.
  ✅  xG máximo = 17.49 — rango plausible.
  ✅  Total shots máximo = 100 — rango plausible.
────────────────────────────────────────────────────────────
  ✅  Sin problemas detectados en bundesliga

## Celda 10 — Verificar resultados y combinar master

In [12]:
import glob

CARPETA_MASTER = "dataset_final"
RUTA_MASTER    = os.path.join(CARPETA_MASTER, "all_leagues_master_v3.csv")

csvs = [f for f in glob.glob(os.path.join(CARPETA_MASTER, "*_master.csv"))
        if "all_leagues" not in f]

if not csvs:
    print("❌ No hay CSVs de liga. Ejecuta la Celda 9 primero.")
else:
    print(f"{'═'*70}")
    print(f"  {'Liga':<25} {'Jugadores':>10} {'Cols':>6}  {'NaN':>8}  {'xG=0':>6}  {'Goals>5':>8}")
    print(f"{'─'*70}")

    dfs = {}
    alertas_globales = []

    for csv_file in sorted(csvs):
        liga_key = os.path.basename(csv_file).replace("_master.csv", "")
        nombre   = LIGAS.get(liga_key, {}).get("nombre", liga_key)
        try:
            df = pd.read_csv(csv_file)
            dfs[liga_key] = df

            n_nan      = df.isnull().sum().sum()
            n_xg_cero  = (df.get("Expected goals (xG)", pd.Series(dtype=float)) == 0).sum() if "Expected goals (xG)" in df.columns else "N/A"
            n_goals_gt = (df.get("Goals", pd.Series(dtype=float)) > 5).sum() if "Goals" in df.columns else "N/A"

            # Alerta si xG = 0 para TODOS (significa que la liga no tiene datos)
            if isinstance(n_xg_cero, int) and n_xg_cero == len(df):
                alertas_globales.append(f"⚠️  {nombre}: xG = 0 para todos. Debe ser NaN.")

            # Alerta si Goals nunca supera 5 (probable dato por partido)
            if isinstance(n_goals_gt, int) and n_goals_gt == 0 and len(df) > 50:
                alertas_globales.append(f"⚠️  {nombre}: ningún jugador tiene Goals > 5. Revisar scraping.")

            estado = "⚠️ " if n_nan > 0 else "✅"
            print(f"  {estado} {nombre:<23} {len(df):>10,} {len(df.columns):>6}  {n_nan:>8,}  {n_xg_cero!s:>6}  {n_goals_gt!s:>8}")

        except Exception as e:
            print(f"  ❌  {nombre:<23}  Error: {e}")

    print(f"{'═'*70}")

    if alertas_globales:
        print("\n⚠️  ALERTAS GLOBALES:")
        for a in alertas_globales:
            print(f"  {a}")

    # Combinar todo en un master
    if dfs:
        df_master = pd.concat(list(dfs.values()), ignore_index=True)
        df_master.to_csv(RUTA_MASTER, index=False)
        print(f"\n✅ Master combinado guardado: {RUTA_MASTER}")
        print(f"   Total jugadores: {len(df_master):,}")
        print(f"   Total columnas : {len(df_master.columns)}")

        # Validación final del master completo
        print("\n🔍 Validación del master completo:")
        validar_merge(df_master, "ALL_LEAGUES")

══════════════════════════════════════════════════════════════════════
  Liga                       Jugadores   Cols       NaN    xG=0   Goals>5
──────────────────────────────────────────────────────────────────────
  ⚠️  2. Bundesliga                  536     35     5,178       0        42
  ⚠️  Bundesliga                     499     35     3,797       0        45
  ⚠️  Championship                   748     35     6,414       8        75
  ⚠️  LaLiga Hypermotion              60     18       308     N/A        60
  ⚠️  LaLiga EA Sports               599     35     4,915       0        59
  ⚠️  Liga Profesional Argentina        819     35     8,950      19         9
  ⚠️  Ligue 1                        551     35     5,159      12        38
  ⚠️  Ligue 2                        536     28     4,352     N/A        33
  ⚠️  MLS                            718     35     6,442       0        31
  ⚠️  Premier League                 537     35     3,815       0        56
  ⚠️  Serie A        

In [13]:
import pandas as pd

def fix_mojibake_text(value):
    if not isinstance(value, str):
        return value
    replacements = {
        "Ã¡": "á",
        "Ã©": "é",
        "Ã­": "í",
        "Ã³": "ó",
        "Ãº": "ú",
        "Ã": "Á",
        "Ã‰": "É",
        "Ã": "Í",
        "Ã“": "Ó",
        "Ãš": "Ú",
        "Ã±": "ñ",
        "Ã‘": "Ñ",
        "Ã¼": "ü",
        "Ãœ": "Ü",
        "â‚¬": "€",
        "Â": "",
    }
    s = value
    for bad, good in replacements.items():
        s = s.replace(bad, good)
    return s.strip()

def normalize_name(name):
    if pd.isna(name):
        return name
    name = str(name).strip()
    name = fix_mojibake_text(name)
    name = " ".join(name.split())  # normaliza espacios dobles
    return name.lower()

# Rutas
csv_1_path = "C:\\Users\\jaime\\OneDrive\\Escritorio\\Proyectos\\TFM_Rayo_Vallecano\\rayo_scouting_web\\backend\\data\\all_leagues_master_v3.csv"
csv_2_path = "C:\\Users\\jaime\\OneDrive\\Escritorio\\Proyectos\\TFM_Rayo_Vallecano\\rayo_scouting_web\\backend\\data\\all_leagues_master_v2.csv"
output_path = "C:\\Users\\jaime\\OneDrive\\Escritorio\\Proyectos\\TFM_Rayo_Vallecano\\rayo_scouting_web\\backend\\data\\all_leagues_master_v3_con_minutes(antiguos).csv"

# Carga
df1 = pd.read_csv(csv_1_path, encoding="utf-8", low_memory=False)
df2 = pd.read_csv(csv_2_path, encoding="utf-8", low_memory=False)

# Crear claves normalizadas
df1["Name_norm"] = df1["Name"].apply(normalize_name)
df2["Name_norm"] = df2["Name"].apply(normalize_name)

# Quedarnos con Name_norm + minutes_played
df2_minutes = df2[["Name_norm", "minutes_played"]].copy()

# Si hubiese duplicados en el segundo CSV, nos quedamos con el mayor minutes_played
df2_minutes = (
    df2_minutes
    .sort_values("minutes_played", ascending=False)
    .drop_duplicates(subset=["Name_norm"], keep="first")
)

# Merge
df_merged = df1.merge(df2_minutes, on="Name_norm", how="left")

# Limpiar columna auxiliar
df_merged = df_merged.drop(columns=["Name_norm"])

# Resumen
found = df_merged["minutes_played"].notna().sum()
missing = df_merged["minutes_played"].isna().sum()

print(f"Jugadores con minutes_played encontrados: {found}")
print(f"Jugadores sin coincidencia: {missing}")

if missing > 0:
    print("\nJugadores sin match:")
    print(df_merged.loc[df_merged["minutes_played"].isna(), "Name"].tolist())

# Guardar
df_merged.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\nArchivo guardado en: {output_path}")

Jugadores con minutes_played encontrados: 6192
Jugadores sin coincidencia: 586

Jugadores sin match:
['Christian Mathenia', 'Daniel Mesenhöler', 'Ron Thorben Hoffmann', 'Enis Kamga', 'Noah Sarenren Bazee', 'Christian Conteh', 'Ben Bobzien', 'Moritz Kwarteng', 'Shinta Appelkamp', 'Florian Yves Le Joncour', 'Tim Oberdorf', 'Merveille Papela', 'Johan Gomez', 'Patrick Nkoa', 'Jano Ter-Horst', 'Luca Itter', 'Santiago Castañeda', 'Ikem Ugoh', 'Lars Raebiger', 'Mikail Demirhan', 'Valgeir Lunddal Fridriksson', 'Ben Jungfleisch', 'Kilian Sauck', 'Andu Kelati', 'Zaid Amoussou-Tchibara', 'Charalampos Makridis', 'Aremu Afeez', 'Niklas Mohr', 'Jan-Hendrik Marx', 'Dylan Leonard', 'Luca Pfeiffer', 'Soufian Gouram', 'Niklas Hildebrandt', 'David Schramm', 'Leon Parduzi', 'Grayson Dettoni', 'Jan Uwe Thielmann', 'Yannick Engelhardt', 'Ayoube Amaimouni Echghouyab', 'Joe Scally', 'Tom Alexander Rothe', 'Kim Min-jae', 'Salim Musah', 'Stanley Nsoki', 'Matheo Raab', 'Sander Tangvik', 'Dan Zagadou', 'Mick Schm

## Mejora de datos

Se mejora el csv de datos añadiendo los minutos de los jugadores y otras stats

In [17]:
import os
import re
import time
import random
import unicodedata
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager


# ============================================================================
# CONFIGURACIÓN
# ============================================================================

OTHER_BUTTON_TEXTS = ["Other", "Otros", "Otro", "Más"]
APPLY_BUTTON_TEXTS = ["Apply", "Aplicar"]

COLUMNAS_A_QUITAR = [
    ["Goals", "Goles"],
    ["Succ. dribbles", "Regates exitosos", "Regates succ."],
    ["Tackles", "Entradas"],
    ["Assists", "Asistencias"],
    ["Accurate passes %", "Pases precisos %", "% pases precisos"],
    ["Average Sofascore Rating", "Media Sofascore Rating", "Rating medio"],
]

EXTRA_COLUMNS = [
    {
        "checkbox_id": "checkbox-minutesPlayed-minutesPlayed",
        "col_name": "minutes_played",
        "header_keywords": ["minutes played", "minutes", "minutos"],
    },
    {
        "checkbox_id": "checkbox-groundDuelsWonPercentage-groundDuelsWonPercentage",
        "col_name": "ground_duels_won_pct",
        "header_keywords": ["ground duels won", "ground", "terrestre", "duelos por bajo"],
    },
    {
        "checkbox_id": "checkbox-aerialDuelsWonPercentage-aerialDuelsWonPercentage",
        "col_name": "aerial_duels_won_pct",
        "header_keywords": ["aerial duels won", "aerial", "aéreo", "aereo", "duelos aéreos"],
    },
    {
        "checkbox_id": "checkbox-totalDuelsWonPercentage-totalDuelsWonPercentage",
        "col_name": "total_duels_won_pct",
        "header_keywords": ["total duels won", "total duel", "total duels", "duelos totales"],
    },
    {
        "checkbox_id": "checkbox-fouls-fouls",
        "col_name": "fouls",
        "header_keywords": ["fouls", "foul", "falta", "faltas"],
    },
]

LIGAS = {
    # "serie_a": {
    #     "nombre": "Serie A",
    #     "url": "https://www.sofascore.com/football/tournament/italy/serie-a/23#id:76457,tab:stats",
    # },
    # "laliga": {
    #     "nombre": "LaLiga EA Sports",
    #     "url": "https://www.sofascore.com/football/tournament/spain/laliga/8#id:77559,tab:stats",
    # },
    "laliga2": {
        "nombre": "LaLiga Hypermotion",
        "url": "https://www.sofascore.com/football/tournament/spain/laliga-2/54#id:77558,tab:stats",
    },
    "premier": {
        "nombre": "Premier League",
        "url": "https://www.sofascore.com/football/tournament/england/premier-league/17#id:76986,tab:stats",
    },
    #"championship": {
    #    "nombre": "Championship",
    #    "url": "https://www.sofascore.com/football/tournament/england/championship/18#id:77347,tab:stats",
    #},
    #"bundesliga": {
    #    "nombre": "Bundesliga",
    #    "url": "https://www.sofascore.com/football/tournament/germany/bundesliga/35#id:77333,tab:stats",
    #},
    # "bundesliga2": {
    #     "nombre": "2. Bundesliga",
    #     "url": "https://www.sofascore.com/football/tournament/germany/2-bundesliga/44#id:77354,tab:stats",
    # },
    # "ligue1": {
    #     "nombre": "Ligue 1",
    #     "url": "https://www.sofascore.com/football/tournament/france/ligue-1/34#id:77356,tab:stats",
    # },
    # "ligue2": {
    #     "nombre": "Ligue 2",
    #     "url": "https://www.sofascore.com/football/tournament/france/ligue-2/182#id:77357,tab:stats",
    # },
    # "serie_b": {
    #     "nombre": "Serie B",
    #     "url": "https://www.sofascore.com/football/tournament/italy/serie-b/53#id:79502,tab:stats",
    # },
    "mls": {
        "nombre": "MLS",
        "url": "https://www.sofascore.com/football/tournament/usa/mls/242#id:70158,tab:stats",
    },
    "liga_argentina": {
        "nombre": "Liga Profesional Argentina",
        "url": "https://www.sofascore.com/football/tournament/argentina/liga-profesional-de-futbol/155#id:87913,tab:stats",
    },
}

OUTPUT_DIR = os.path.join("dataset_final", "minutes_temp")
os.makedirs(OUTPUT_DIR, exist_ok=True)

FINAL_OUTPUT = os.path.join("dataset_final", "fresh_minutes_and_extra.csv")


# ============================================================================
# HELPERS
# ============================================================================

def normalizar_nombre(nombre: str) -> str:
    if not isinstance(nombre, str):
        return ""
    nombre = nombre.strip().lower()
    nombre = unicodedata.normalize("NFD", nombre)
    nombre = "".join(c for c in nombre if unicodedata.category(c) != "Mn")
    nombre = re.sub(r"\s+", " ", nombre)
    return nombre

def espera_humana(min_seg=1.0, max_seg=3.0):
    time.sleep(random.uniform(min_seg, max_seg))

def _click_robusto(driver, elemento):
    try:
        elemento.click()
    except Exception:
        driver.execute_script("arguments[0].click();", elemento)

def _encontrar_boton_por_textos(driver, textos: list, scope=None):
    base = scope if scope else driver
    for texto in textos:
        for xpath in [
            f".//button[normalize-space(.)='{texto}']",
            f".//button[contains(normalize-space(.),'{texto}')]",
        ]:
            try:
                return base.find_element(By.XPATH, xpath)
            except NoSuchElementException:
                continue
    return None

def crear_driver(headless=True):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--lang=en-US")
    options.add_experimental_option(
        "prefs", {
            "intl.accept_languages": "en-US,en",
            "translate_whitelists": {},
            "translate": {"enabled": False},
        }
    )
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=options
    )
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver

def cerrar_cookies(driver):
    time.sleep(2)
    for sel in [
        "button.fc-button.fc-cta-consent.fc-primary-button",
        "button.fc-button.fc-cta-do-not-consent",
        "button[aria-label='Consent']",
        ".fc-cta-consent",
        "#onetrust-accept-btn-handler",
    ]:
        try:
            btn = driver.find_element(By.CSS_SELECTOR, sel)
            if btn.is_displayed():
                _click_robusto(driver, btn)
                espera_humana(1, 2)
                print("    ✅ Cookies aceptadas")
                return
        except NoSuchElementException:
            continue

def cerrar_overlays(driver):
    for sel in [".fc-dialog-overlay", ".fc-dialog-container", ".modal-backdrop", ".modal-overlay", "[role='dialog']"]:
        for el in driver.find_elements(By.CSS_SELECTOR, sel):
            try:
                driver.execute_script("arguments[0].remove();", el)
            except Exception:
                pass

def cerrar_modal_idioma(driver, wait):
    textos_modal = [
        "Is this the right language?", "Looks like you", "Do you want to continue",
        "¿Es este el idioma correcto?", "Parece que estás en", "¿Quieres continuar",
    ]
    try:
        for texto in textos_modal:
            try:
                wait.until(EC.presence_of_element_located((By.XPATH, f"//*[contains(text(),'{texto}')]")))
                break
            except Exception:
                continue

        for texto_btn in ["Confirm", "Continue", "OK", "Aceptar", "Confirmar", "Continuar"]:
            try:
                btn = driver.find_element(By.XPATH, f"//button[normalize-space(.)='{texto_btn}']")
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
                _click_robusto(driver, btn)
                espera_humana(1, 2)
                print("    ✅ Modal idioma cerrado")
                return True
            except NoSuchElementException:
                continue
    except Exception:
        pass
    return False

def preparar_tabla_detailed(driver, wait, liga_key: str):
    print("    ⏳ Esperando panel de stats...")

    try:
        wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="tabpanel-stats"]')))
        espera_humana(2, 3)
        print("    ✅ Panel stats cargado")
    except Exception as e:
        print(f"    ❌ Panel stats no encontrado: {e}")
        return False

    # Activar detailed
    detailed_selectors = [
        (By.CSS_SELECTOR, 'button[data-testid="tab-detailed"]'),
        (By.XPATH, "//button[contains(., 'Detailed')]"),
    ]

    detailed_ok = False
    for by, sel in detailed_selectors:
        try:
            btn = wait.until(EC.element_to_be_clickable((by, sel)))
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            _click_robusto(driver, btn)
            espera_humana(2, 3)
            detailed_ok = True
            print("    ✅ Pestaña Detailed activada")
            break
        except Exception:
            continue

    if not detailed_ok:
        print("    ❌ No se pudo activar Detailed")
        return False

    # Eliminar columnas visibles no deseadas
    print("    🗑️ Eliminando columnas por defecto...")
    for variantes in COLUMNAS_A_QUITAR:
        eliminado = False
        for nombre_col in variantes:
            try:
                btn_x = driver.find_element(
                    By.XPATH,
                    f"//button[contains(@class,'button--variant_filled') "
                    f"and contains(@class,'button--size_tertiary') "
                    f"and starts-with(normalize-space(.), '{nombre_col}')]"
                )
                _click_robusto(driver, btn_x)
                espera_humana(0.3, 0.6)
                print(f"      • eliminada: {nombre_col}")
                eliminado = True
                break
            except NoSuchElementException:
                continue
        if not eliminado:
            print(f"      ℹ️ no encontrada para eliminar: {variantes[0]}")

    espera_humana(1, 1.5)

    # Abrir menú Other
    print("    🔧 Abriendo menú Other...")
    btn_other = _encontrar_boton_por_textos(driver, OTHER_BUTTON_TEXTS)
    if btn_other is None:
        print("    ❌ Botón Other no encontrado")
        return False

    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn_other)
    _click_robusto(driver, btn_other)
    espera_humana(1.5, 2.5)
    print("    ✅ Menú Other abierto")

    # Activar checkboxes
    print("    ☑️ Activando columnas extra...")
    activados = 0
    for col_cfg in EXTRA_COLUMNS:
        try:
            checkbox = wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, f"input#{col_cfg['checkbox_id']}"))
            )
            if not checkbox.is_selected():
                label = driver.find_element(By.CSS_SELECTOR, f"label[for='{col_cfg['checkbox_id']}']")
                driver.execute_script("arguments[0].click();", label)
                espera_humana(0.3, 0.6)
            activados += 1
            print(f"      • activada: {col_cfg['col_name']}")
        except Exception as e:
            print(f"      ⚠️ no se pudo activar {col_cfg['col_name']}: {type(e).__name__}")

    espera_humana(0.5, 1.0)

    # Apply
    print("    📤 Aplicando cambios...")
    apply_ok = False
    for texto in APPLY_BUTTON_TEXTS:
        try:
            btn_apply = wait.until(EC.element_to_be_clickable(
                (By.XPATH,
                 f"//button[contains(@class,'button--variant_filled') "
                 f"and contains(@class,'button--size_primary') "
                 f"and normalize-space(.)='{texto}']")
            ))
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn_apply)
            _click_robusto(driver, btn_apply)
            espera_humana(2, 3)
            print(f"    ✅ '{texto}' pulsado")
            apply_ok = True
            break
        except Exception:
            continue

    if not apply_ok:
        print("    ⚠️ No se encontró botón Apply/Aplicar")

    return activados > 0

def extraer_tabla_extra(driver, wait):
    TABLA_XPATH = (
        '//*[@id="tabpanel-detailed"]//table'
        ' | //*[@id="tabpanel-stats"]//table'
        ' | //*[@id="tabpanel-summary"]//table'
    )

    try:
        tabla = wait.until(EC.presence_of_element_located((By.XPATH, TABLA_XPATH)))

        encabezados = []
        for th in tabla.find_elements(By.TAG_NAME, "th"):
            titulo = th.get_attribute("title") or th.text.strip()
            encabezados.append(titulo.strip() if titulo else "")

        enc_lower = [h.lower() for h in encabezados]
        print(f"    🧾 Headers detectados: {encabezados}")

        idx_name = next((i for i, h in enumerate(enc_lower) if h in ("name", "nombre", "player")), None)

        idx_extra = {}
        for col_cfg in EXTRA_COLUMNS:
            idx = None
            for kw in col_cfg["header_keywords"]:
                matches = [i for i, h in enumerate(enc_lower) if kw in h]
                if matches:
                    idx = matches[0]
                    break
            idx_extra[col_cfg["col_name"]] = idx

        print(f"    🔎 Índices detectados: name={idx_name}, extras={idx_extra}")

        filas = []
        tbody = tabla.find_element(By.TAG_NAME, "tbody")

        for fila in tbody.find_elements(By.TAG_NAME, "tr"):
            celdas = fila.find_elements(By.TAG_NAME, "td")
            if not celdas:
                continue

            nombre = ""
            if idx_name is not None and idx_name < len(celdas):
                nombre = celdas[idx_name].text.strip()

            if not nombre:
                continue

            fila_dict = {"Name": nombre}

            for col_cfg in EXTRA_COLUMNS:
                col_name = col_cfg["col_name"]
                idx = idx_extra.get(col_name)
                valor = ""
                if idx is not None and idx < len(celdas):
                    valor = celdas[idx].text.strip()
                fila_dict[col_name] = valor

            filas.append(fila_dict)

        return filas

    except Exception as e:
        print(f"    ❌ Error extrayendo tabla: {e}")
        return []

def hay_siguiente_pagina(driver):
    PANEL_IDS = ["tabpanel-detailed", "tabpanel-stats", "tabpanel-summary"]

    try:
        btn_sig = None
        for panel_id in PANEL_IDS:
            try:
                paginador = driver.find_element(By.XPATH, f'//*[@id="{panel_id}"]/div/div')
                botones = paginador.find_elements(By.CSS_SELECTOR, "button.button--size_primary")
                if botones:
                    btn_sig = botones[-1]
                    break
            except NoSuchElementException:
                continue

        if btn_sig is None:
            return False

        if btn_sig.get_attribute("disabled") is not None:
            return False

        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn_sig)
        _click_robusto(driver, btn_sig)
        espera_humana(1.5, 2.5)
        return True

    except Exception:
        return False


# ============================================================================
# SCRAPER PRINCIPAL
# ============================================================================

def scrapear_extra_stats_liga(liga_key: str, headless: bool = True, intento: int = 1) -> pd.DataFrame:
    cfg = LIGAS[liga_key]
    driver = crear_driver(headless)
    wait = WebDriverWait(driver, 25)
    todos = []

    try:
        print(f"\n{'='*80}")
        print(f"🏆 {cfg['nombre']} ({liga_key})")
        print(f"{'='*80}")

        print(f"    📍 Navegando a {cfg['url']}")
        driver.get(cfg["url"])
        espera_humana(4, 6)

        cerrar_cookies(driver)
        cerrar_overlays(driver)
        cerrar_modal_idioma(driver, wait)
        espera_humana(2, 3)

        ok = preparar_tabla_detailed(driver, wait, liga_key)
        if not ok:
            print(f"    ❌ No se pudo preparar la tabla detailed para {liga_key}")
            return pd.DataFrame(columns=["Name"] + [c["col_name"] for c in EXTRA_COLUMNS])

        num_pagina = 1
        while True:
            print(f"    → Página {num_pagina}...")

            filas = extraer_tabla_extra(driver, wait)
            print(f"      {len(filas)} jugadores extraídos")
            todos.extend(filas)

            if not hay_siguiente_pagina(driver):
                print(f"    ✅ Última página alcanzada ({num_pagina})")
                break

            num_pagina += 1

    except Exception as e:
        print(f"    ❌ Error general en {liga_key}: {e}")

    finally:
        driver.quit()

    df = pd.DataFrame(todos)

    if not df.empty:
        df["_name_norm"] = df["Name"].apply(normalizar_nombre)
        if "minutes_played" in df.columns:
            df["minutes_played_num"] = pd.to_numeric(
                df["minutes_played"].astype(str).str.replace(".", "", regex=False).str.replace(",", "", regex=False),
                errors="coerce"
            )
            df = (
                df.sort_values("minutes_played_num", ascending=False, na_position="last")
                  .drop_duplicates(subset="_name_norm", keep="first")
                  .drop(columns=["minutes_played_num"])
            )
        else:
            df = df.drop_duplicates(subset="_name_norm", keep="first")

        df = df.drop(columns=["_name_norm"])

    return df


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_dfs = []

    for liga_key in LIGAS:
        df_liga = scrapear_extra_stats_liga(liga_key, headless=False)

        out_path = os.path.join(OUTPUT_DIR, f"{liga_key}_minutes_extra.csv")
        df_liga.to_csv(out_path, index=False, encoding="utf-8-sig")
        print(f"    💾 Guardado: {out_path} ({len(df_liga)} filas)")

        if not df_liga.empty:
            df_liga["liga"] = liga_key
            all_dfs.append(df_liga)

    if all_dfs:
        df_final = pd.concat(all_dfs, ignore_index=True)
        df_final.to_csv(FINAL_OUTPUT, index=False, encoding="utf-8-sig")
        print(f"\n✅ CSV final generado: {FINAL_OUTPUT}")
        print(f"   Filas: {len(df_final)}")
        print(f"   Columnas: {df_final.columns.tolist()}")
    else:
        print("\n❌ No se extrajeron datos de ninguna liga")


if __name__ == "__main__":
    main()

2026-06-12 18:02:27,610 | INFO | ====== WebDriver manager ======
2026-06-12 18:02:33,067 | INFO | Get LATEST chromedriver version for google-chrome
2026-06-12 18:02:34,183 | INFO | Get LATEST chromedriver version for google-chrome
2026-06-12 18:02:35,058 | INFO | Driver [C:\Users\jaime\.wdm\drivers\chromedriver\win64\149.0.7827.115\chromedriver-win32/chromedriver.exe] found in cache



🏆 LaLiga Hypermotion (laliga2)
    📍 Navegando a https://www.sofascore.com/football/tournament/spain/laliga-2/54#id:77558,tab:stats
    ✅ Cookies aceptadas
    ✅ Modal idioma cerrado
    ⏳ Esperando panel de stats...
    ✅ Panel stats cargado
    ✅ Pestaña Detailed activada
    🗑️ Eliminando columnas por defecto...
      • eliminada: Goles
      ℹ️ no encontrada para eliminar: Succ. dribbles
      • eliminada: Entradas
      • eliminada: Asistencias
      • eliminada: Pases precisos %
      ℹ️ no encontrada para eliminar: Average Sofascore Rating
    🔧 Abriendo menú Other...
    ✅ Menú Other abierto
    ☑️ Activando columnas extra...
      • activada: minutes_played
      • activada: ground_duels_won_pct
      • activada: aerial_duels_won_pct
      • activada: total_duels_won_pct
      • activada: fouls
    📤 Aplicando cambios...
    ✅ 'Aplicar' pulsado
    → Página 1...
    🧾 Headers detectados: ['#', 'Equipos', 'Nombre', 'Regates completados', 'Valoración Sofascore promedio', 'Minut

2026-06-12 18:11:22,259 | INFO | ====== WebDriver manager ======


    💾 Guardado: dataset_final\minutes_temp\laliga2_minutes_extra.csv (672 filas)


2026-06-12 18:11:25,665 | INFO | Get LATEST chromedriver version for google-chrome
2026-06-12 18:11:26,635 | INFO | Get LATEST chromedriver version for google-chrome
2026-06-12 18:11:27,409 | INFO | Driver [C:\Users\jaime\.wdm\drivers\chromedriver\win64\149.0.7827.115\chromedriver-win32/chromedriver.exe] found in cache


...

2026-06-12 18:30:10,066 | WARNING | Retrying (Retry(total=2, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPConnectionPool(host='localhost', port=56380): Read timed out. (read timeout=120)")': /session/b82140215a41c0cfb3eb843b84b3e35b
2026-06-12 18:32:10,114 | WARNING | Retrying (Retry(total=1, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPConnectionPool(host='localhost', port=56380): Read timed out. (read timeout=120)")': /session/b82140215a41c0cfb3eb843b84b3e35b
2026-06-12 18:34:10,146 | WARNING | Retrying (Retry(total=0, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPConnectionPool(host='localhost', port=56380): Read timed out. (read timeout=120)")': /session/b82140215a41c0cfb3eb843b84b3e35b
2026-06-12 18:36:24,318 | INFO | ====== WebDriver manager ======


    💾 Guardado: dataset_final\minutes_temp\mls_minutes_extra.csv (160 filas)


2026-06-12 18:36:37,266 | INFO | Get LATEST chromedriver version for google-chrome
2026-06-12 18:36:39,213 | INFO | Get LATEST chromedriver version for google-chrome
2026-06-12 18:36:41,474 | INFO | Driver [C:\Users\jaime\.wdm\drivers\chromedriver\win64\149.0.7827.115\chromedriver-win32/chromedriver.exe] found in cache



🏆 Liga Profesional Argentina (liga_argentina)
    📍 Navegando a https://www.sofascore.com/football/tournament/argentina/liga-profesional-de-futbol/155#id:87913,tab:stats


KeyboardInterrupt: 

In [22]:
import pandas as pd
import glob
import os

# --- CONFIGURACIÓN ---
CARPETA_CSVS = "C:\\Users\\jaime\\OneDrive\\Escritorio\\Proyectos\\TFM_Rayo_Vallecano\\rayo_scouting_web\\notebooks\\dataset_final\\minutes_temp"          # carpeta donde están todos los csvs por liga
ARCHIVO_MASTER = "C:\\Users\\jaime\\OneDrive\\Escritorio\\Proyectos\\TFM_Rayo_Vallecano\\rayo_scouting_web\\backend\\data\\all_leagues_master_v3.csv"     # tu csv con todos los jugadores
ARCHIVO_SALIDA = "C:\\Users\\jaime\\OneDrive\\Escritorio\\Proyectos\\TFM_Rayo_Vallecano\\rayo_scouting_web\\backend\\data\\all_leagues_master_v4.csv"   # csv resultante

# Columnas que vienen de los csvs "minutes"
COLUMNAS_MINUTES = ["Name", "minutes_played", "ground_duels_won_pct",
                     "aerial_duels_won_pct", "total_duels_won_pct", "fouls"]

# --- 1. Buscar los csvs que tengan "minutes" en el nombre ---
patron = os.path.join(CARPETA_CSVS, "*minutes*.csv")
archivos_minutes = glob.glob(patron)

print(f"Archivos encontrados con 'minutes': {len(archivos_minutes)}")
for f in archivos_minutes:
    print(" -", f)

# --- 2. Leer y concatenar todos esos csvs ---
dfs = []
for archivo in archivos_minutes:
    df = pd.read_csv(archivo)
    dfs.append(df[COLUMNAS_MINUTES])

df_minutes = pd.concat(dfs, ignore_index=True)

# Por si un jugador aparece repetido entre varios csvs (p.ej. mismo nombre
# en distintas ligas), nos quedamos con la primera aparición.
# Si prefieres que se queden TODAS y luego decidir, comenta esta línea.
df_minutes = df_minutes.drop_duplicates(subset="Name", keep="first")

# --- 3. Leer el csv master ---
df_master = pd.read_csv(ARCHIVO_MASTER)

# --- 4. Unir por nombre (left join: mantiene todos los del master) ---
df_final = df_master.merge(df_minutes, on="Name", how="left")

# --- 5. Guardar resultado ---
df_final.to_csv(ARCHIVO_SALIDA, index=False)

print(f"\nListo. Guardado en: {ARCHIVO_SALIDA}")
print(f"Filas master: {len(df_master)}, Filas tras merge: {len(df_final)}")

# Comprobación rápida: jugadores del master que NO encontraron match
sin_match = df_final[df_final["minutes_played"].isna()]
print(f"Jugadores sin datos de 'minutes': {len(sin_match)}")

Archivos encontrados con 'minutes': 12
 - C:\Users\jaime\OneDrive\Escritorio\Proyectos\TFM_Rayo_Vallecano\rayo_scouting_web\notebooks\dataset_final\minutes_temp\bundesliga2_minutes_extra.csv
 - C:\Users\jaime\OneDrive\Escritorio\Proyectos\TFM_Rayo_Vallecano\rayo_scouting_web\notebooks\dataset_final\minutes_temp\bundesliga_minutes_extra.csv
 - C:\Users\jaime\OneDrive\Escritorio\Proyectos\TFM_Rayo_Vallecano\rayo_scouting_web\notebooks\dataset_final\minutes_temp\championship_minutes_extra.csv
 - C:\Users\jaime\OneDrive\Escritorio\Proyectos\TFM_Rayo_Vallecano\rayo_scouting_web\notebooks\dataset_final\minutes_temp\laliga2_minutes_extra.csv
 - C:\Users\jaime\OneDrive\Escritorio\Proyectos\TFM_Rayo_Vallecano\rayo_scouting_web\notebooks\dataset_final\minutes_temp\laliga_minutes_extra.csv
 - C:\Users\jaime\OneDrive\Escritorio\Proyectos\TFM_Rayo_Vallecano\rayo_scouting_web\notebooks\dataset_final\minutes_temp\liga_argentina_minutes_extra.csv
 - C:\Users\jaime\OneDrive\Escritorio\Proyectos\TFM_Ray